In [1]:
import base64
import os
from mistralai import Mistral
from dotenv import load_dotenv
from pathlib import Path
from fill_pa import fill_pa
from pydantic import BaseModel
# from get_pa_fields import get_pa_fields
import time
from google import genai 
from google.genai import types
import pymupdf
import json
from typing import Union

load_dotenv()

True

In [2]:
#define pydantic reponse model for gemini api
class Field(BaseModel):
    id: str
    value: Union[str, bool]

In [3]:
def encode_pdf(pdf_path):
    """Encode the pdf to base64."""
    try:
        with open(pdf_path, "rb") as pdf_file:
            return base64.b64encode(pdf_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {pdf_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

In [4]:

# Path to your referral package pdf
script_dir = Path(".").parent
pdf_path = script_dir / ".." /"Input Data" /"Adbulla" / "referral_package.pdf"
pdf_path = str(pdf_path.resolve())

In [5]:
#Path to pA pdf
file_path = script_dir / ".." /"Input Data" /"Adbulla" / "PA.pdf"

#pass the pdf to the fill_pa file to annotate the widgets in the PA
fill_pa(file_path)

#path to new annotated PA pdf
file_path = script_dir /"PA_edited.pdf"


In [6]:
# Getting the base64 string to send to mistral api
base64_pdf = encode_pdf(pdf_path)

#instantiate mistral
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

#instantiate gemini
gemini_key = os.environ["GEMINI_API_KEY"]
gemini_client = genai.Client(api_key=gemini_key)

In [7]:

#read referral package with mistral ocr
ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "document_url",
        "document_url": f"""data:application/pdf;base64,{base64_pdf}""" 
    },
    include_image_base64=True
)

In [8]:
#add referral package information to user prompt variable 
prompt = ""
for page in ocr_response.pages:
    prompt += page.markdown
    
pdf_path = script_dir / ".." /"Input Data" /"Adbulla" / "PA.pdf"
print("Added referral info to prompt")

Added referral info to prompt


In [9]:
#query the gemini llm with pa field names and referral package information embedded in instructions and user prompt respectively.
chat_response = gemini_client.models.generate_content(
    model = "gemini-2.0-flash",
    config=types.GenerateContentConfig(
    system_instruction= f"""You are an expert medical data extractor. Your task is to accurately extract information from a provided medical record and fill out the pdf attached. Pay close attention to dates, patient demographics, medical history, and medication details.

**Instructions:**

1.  **Checkboxes:** If a field is a checkbox that has to be checked based on information from pdf, the value of the field is True otherwise False. Don't use the field's name as the value
2.  **Date Format:** All dates should be in MM/DD/YYYY format. If only partial date information is available leave blank the missing ones.
4.  **Drug Lists:** For sections with lists of drugs, if a drug is mentioned as being used, failed, or causing an adverse reaction, identify that specific drug. If multiple drugs are mentioned for a single choice (e.g., "Riabni (rituximab-arrx) Rituxan (rituximab)"), select only the relevant one.
5.  **Descriptive Fields:** For fields requiring descriptions (e.g., "Please describe the nature of the failure of the preferred drug"), extract the relevant text directly from the medical record.
6.  **"Other" Fields:** If "Other" is a selectable option, provide the specific "Other" value if present in the medical record.
7.  **Measurements:** Ensure weights are in lbs or kgs and heights in inches or cms, as indicated by the field. Convert if necessary or note if units are different.
8.  **Empty Fields:** If a field is not found or cannot be inferred from the medical record, leave its value blank. Do not invent information.
9.  **Field Id:** Every field has a name annotated to it(on top of checkboxes if field is textbox and in the textarea if field is a text area), in your response, use the index as the index field and then the set the value using the information in the medical record as reference. This indices will help a form filling software to find and populate the approriate fields.
10. **Text Fields:** Text fields do not take a yes or no answer, output rather the necessary information based on medical record or leave blank if you cannot find helpful information from the medication records. All other fields except checkboxes are text fields.
Don't infer the names instead use the names as provided in the document. 
You can infer some checkbox values based on information available
**Extract the corresponding value from the medical record for each field and return them as a list following the schema. The medical record will be supplied in the user prompt:**
            """,
    response_mime_type="application/json",
    response_schema=list[Field]
            ),
    contents = [
        types.Part.from_bytes(
            data=file_path.read_bytes(),
            mime_type="application/pdf"
        ),
        prompt
        ]
)


In [10]:
#write output to a file which will be read from later to populate the pa

output = open("./field_info.json", "w+")

output.write(chat_response.text)

print(chat_response.text)

[
  {
    "id": "CB5",
    "value": ""
  },
  {
    "id": "T2",
    "value": ""
  },
  {
    "id": "T3",
    "value": ""
  },
  {
    "id": "T4",
    "value": ""
  },
  {
    "id": "T9",
    "value": ""
  },
  {
    "id": "T10",
    "value": ""
  },
  {
    "id": "T11",
    "value": ""
  },
  {
    "id": "T12",
    "value": "Shakh"
  },
  {
    "id": "T13",
    "value": "Abdulla"
  },
  {
    "id": "T14",
    "value": "04/01/2001"
  },
  {
    "id": "T15",
    "value": "425 Sherman Ave"
  },
  {
    "id": "T16",
    "value": "Nashville"
  },
  {
    "id": "T17",
    "value": "TN"
  },
  {
    "id": "T18",
    "value": "37995"
  },
  {
    "id": "T19",
    "value": "865-395-3958"
  },
  {
    "id": "T20",
    "value": ""
  },
  {
    "id": "T21",
    "value": "865-395-0481"
  },
  {
    "id": "T22",
    "value": ""
  },
  {
    "id": "T23",
    "value": "No Known Allergies"
  },
  {
    "id": "T24",
    "value": ""
  },
  {
    "id": "T25",
    "value": ""
  },
  {
    "id": "T26",
    

In [11]:
with open('field_info.json', 'r+') as file:
    data = json.load(file)
    


In [12]:
#extract field values and populate PA

id_and_values = {elem['id']: elem['value'] for elem in data}
with pymupdf.open('../Input Data/Adbulla/PA.pdf') as source:
    for page in source.pages():
        for widget in page.widgets():
            if widget.field_name in id_and_values:
                widget.field_value = id_and_values[widget.field_name]
                print(type(widget.field_value))
                widget.update()
            
    source.save("PA_Filled.pdf")
    


    
print("success 😎")

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'bool'>
<class 'bool'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'bool'>
<class 'bool'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'bool'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<